# Imagecollections

Στην τρέχουσα ενότητα, δείχνονται μέθοδοι, φιλτραρίσματος και οπτικοποίησης των imagecollections, masking, bit decoding & flag operations, ανάκτηση και δημιουργία νέων ιδιοτήτων, επιλογή και δημιουργία νέων καναλιών (bands). Επιπλέον επιδεικνύεται πως γίνεται η αποκοπή των εικόνων μιας συλλογής με βάση μια γεωμετρία, πως δημιουργείται ένα composite και εκτελούμε πράξεις μεταξύ μπαντών και υπολογίζουμε δείκτες (NDVI) και πως εξάγουμε τις εικόνες στο google drive ή σε ένα τοπικό σύστημα.

In [ ]:
import ee # Google Earth Engine Python Client
import geemap # frontend του ee Google Earth Engine
import pprint # pretty print
from pathlib import Path # διαχείριση αρχείων
import random # για την δημιουργία τυχαίων αριθμών (π.χ. για χρώματα)
import datetime # για την διαχείριση ημερομηνιών
import os # για την διαχείριση περιβάλλοντος και αρχείων
from dotenv import load_dotenv # για την φόρτωση μεταβλητών περιβάλλοντος από .env αρχεία

In [ ]:
# έλεγχος αν το notebook τρέχει στο google colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Colab\ Notebooks/gee
except:
  print("Παρακαλώ εκτελέστε το notebook στο Google Colab")

In [ ]:

# Load environment variables from .env file
load_dotenv()
project_id = os.getenv('project_id')


# gee authentication
ee.Authenticate()
ee.Initialize(project=project_id)



OUTPUT_DIR = Path("output")


In [ ]:
# Feature collection με τα NUTS2 του 2021
nuts_level2 = ee.FeatureCollection('projects/civic-meridian-417810/assets/NUTS_RG_01M_2021_4326_LEVL_2')


# Φιλτράρισμα για μια συγκεκριμένη περιοχή (π.χ. Αττική - EL30)
attiki_region = nuts_level2.filter(ee.Filter.eq('NUTS_ID', 'EL30'))

# πλαίσιο των ορίων της περιοχής
bbox = attiki_region.bounds()


# ImageCollections

Στο google earth engine  το ImageCollection είναι μια συλλογή ή λίστα από εικόνες (Images).

Το ImageCollection μας επιτρέπει να:

- Φιλτράρουμε δεδομένα: Μπορούμε να επιλέξουμε εικόνες βάσει γεωγραφικής περιοχής (filterBounds), χρονικού διαστήματος (filterDate) ή μεταδεδομένων (π.χ. ποσοστό νεφοκάλυψης).

- Επεξεργαζόμαστε μαζικά (Reduce): Αντί να επεξεργαζόμαστε μία-μία τις εικόνες, μπορούμε να εφαρμόσουμε αλγόριθμους σε ολόκληρη τη συλλογή. Για παράδειγμα, μπορείς να ζητήσεις τη "διάμεση τιμή" όλων των εικόνων μιας περιοχής για έναν ολόκληρο χρόνο, δημιουργώντας ένα καθαρό ψηφιακό μωσαϊκό (median composite) χωρίς σύννεφα.

Το ImageCollection αποτελεί συνήθως μια χρονοσειρά δορυφορικών εικόνων από τον ίδιο αισθητήρα. Το GEE δεν φορτώνει όλες τις εικόνες στη μνήμη του τοπικού υπολογιστή. Αντίθετα, "τρέχει" τον υπολογισμό μόνο όταν του ζητήσεις να εμφανίσει το αποτέλεσμα ή να εξάγει τα δεδομένα.


In [ ]:
# Ορισμός της Συλλογής USGS Landsat 8 Level 2, Collection 2, Tier 1 και Φιλτράρισμα
L8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") # https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
              .filterBounds(bbox) # Φιλτράρισμα με βάση τα όρια της Αττικής
              .filterDate('2024-03-01', '2025-03-01') # Φιλτράρισμα με βάση το εύρος ημερομηνιών
           ) 

# Έλεγχος αν η συλλογή είναι κενή πριν προχωρήσουμε
# Εκτύπωση του αριθμού των εικόνων (Size)
count = L8.size().getInfo()
print(f"Συνολικός αριθμός εικόνων στη συλλογή: {count}")

In [ ]:
# πρώτη εικόνα για προβολή όλων των διαθέσιμων ιδιότητων (Metadata keys)
first_image = L8.first()
props = first_image.propertyNames().getInfo()
print("\nΔιαθέσιμες Ιδιότητες (Metadata Properties):")
pprint.pprint(props)


Ο χρήστης έχει την δυνατότητα να προσθέσει νέες ιδιότητες σε κάθε εικόνα του ImageCollection. Κάνοντας χρήση του map και εφαρμόζοντας μια συνάρτηση. Στο παρακάτω παράδειγμα με βάση το property [system:time_start](https://developers.google.com/earth-engine/glossary#system:time_start) μπορούμε να υπολογίζουμε την ημερομηνία και να την προσθέσουμε σαν ένα νέο property με το όνομα year. Στο Google Earth Engine, το system:time_start είναι μια σημαντική μεταβλητή (metadata) για την ανάλυση χρονοσειρών και αφορά τον χρόνο λήψης μιας εικόνας. Η σήμανση γίνεται με τον αριθμό των millisecond που έχουν περάσει από την 1η Ιανουαρίου 1970 (γνωστή ως Unix Epoch).

In [ ]:
def add_year(image):
    # Get the date from system:time_start
    date = ee.Date(image.get('system:time_start')) # Το system:time_start είναι Unix Timestamp. Με την ee.Date() μετατρέπεται σε ημερομηνία που μπορούμε να εξάγουμε το έτος.
    year =  date.get('year') # εξαγωγή του έτους
    return image.set('year',year) # προσθήκη της ιδιότητας 'year' στην εικόνα με την τιμή του έτους

# εφαρμογή της συνάρτησης add_year σε κάθε εικόνα της συλλογής L8
L8 = L8.map(add_year)

Στο παρακάτω παράδειγμα εκτυπώνουμε την λίστα των εικόνων της συλλογής με τις ιδιότητες ID, Ημερομηνία, Έτος και Ποσοστό Νεφοκάλυψης.
Για να γίνει αυτό, πρώτα εξάγουμε τις τιμές των ιδιοτήτων σε λίστες με την [aggregate_array](https://developers.google.com/earth-engine/apidocs/ee-imagecollection-aggregate_array) και getInfo() για να τις μετατρέψουμε σε Python lists. 
Στη συνέχεια, χρησιμοποιούμε ένα for loop για να εκτυπώσουμε κάθε εικόνα με τις αντίστοιχες ιδιότητες.

In [ ]:

# Εκτύπωση συγκεκριμένων ιδιοτήτων για ΟΛΕΣ τις εικόνες της συλλογής
# Παίρνουμε τα ID και το ποσοστό νεφοκάλυψης σε λίστες
ids = L8.aggregate_array('LANDSAT_PRODUCT_ID').getInfo() 
clouds = L8.aggregate_array('CLOUD_COVER').getInfo()
dates = L8.aggregate_array('DATE_ACQUIRED').getInfo()
years = L8.aggregate_array('year').getInfo()


print("\nΛίστα Εικόνων στη Συλλογή:")
for i in range(count):
    print(f"[{i+1}] ID: {ids[i]} | Ημερομηνία: {dates[i]} | Έτος: {years[i]} | Σύννεφα: {clouds[i]}%")


In [ ]:
# Φιλτράρισμα για εικόνες με νεφοκάλυψη > 50%
clear_L8 = L8.filter(ee.Filter.lt('CLOUD_COVER', 50))

print(f"Συνολικός αριθμός εικόνων στη συλλογή: {L8.size().getInfo()}")
print(f"Συνολικός αριθμός εικόνων στη συλλογή (χαμηλότερη νεφοκάλυψη): {clear_L8.size().getInfo()}")

# Φιλτράρισμα για εικόνες με SUN_ELEVATION > 35
clear_L8 = clear_L8.filter(ee.Filter.gt('SUN_ELEVATION', 35))
print(f"Συνολικός αριθμός εικόνων στη συλλογή (χαμηλότερη νεφοκάλυψη+SUN_ELEVATION > 35): {clear_L8.size().getInfo()}")

In [ ]:
# συνάρτηση για εφαρμογή των συντελεστών (scale factors) για τα οπτικά και θερμικά κανάλια
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )

# Εφαρμογή των συντελεστών κλίμακας (scale factors) στην εικόνα του διάμεσου
L8_scaled =  clear_L8.map(apply_scale_factors)

# Εφαρμογή φίλτρο με βάση το QA_PIXEL
Η παρακάτω διαδικασία επιστρέφει μια mask (1=valid, 0=invalid data) με βάση την QA_PIXEL Band, η οποία περιλαμβάνει ποιοτικά στατιστικά στοιχεία για την εικόνα και τα νέφη που την επηρεάζουν. Στόχος είναι η απόκρυψη των pixels που περιέχουν χιόνια, νερό, νέφη και σκιές νεφών καθώς και pixel που έχουν υποστεί ραδιομετρικό κορεσμό κατά την καταγραφή ή που έχουν σημανθεί ως κενά.

Περισσότερες λεπτομέρειες για τα bits flags της μπάντας QA_PIXEL του LandSat 8 μπορείτε να δείτε:
- [Bitmask for QA_PIXEL:](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2#bands)
- [Πληροφορίες για την  Quality Assessment (QA_PIXEL) band](https://www.usgs.gov/media/files/landsat-8-9-collection-2-level-2-science-product-guide)
- παράδειγμα αποκωδικοποίησης στο [επισυναπτόμενο αρχείο](resources/bit_mask.ods)
 

In [ ]:

def maskL8sr(image):
  # Bitmasks για την band QA_PIXEL
  FillBitMask = 1 << 0
  DilatedCloudBitMask = 1 << 1
  cloudsBitMask = 1 << 3
  cloudShadowBitMask = 1 << 4
  snowsBitMask = 1 << 5
  WaterBitMask = 1 << 7
  
  # λήψη της band QA_PIXEL που περιέχει τις πληροφορίες για τα σύννεφα, τη σκιά σύννεφου, το χιόνι, το νερό και τα έγκυρα δεδομένα
  qa = image.select('QA_PIXEL')

  # όλα τα bitmask πρέπει να είναι 0 για να θεωρηθεί το pixel καθαρό (clear)
  # αποδίδειται ένα mask όπου τα pixels που έχουν κάποιο από τα bitmask ενεργοποιημένα (δηλαδή δεν είναι καθαρά) θα έχουν τιμή 0 
  # και τα καθαρά pixels θα έχουν τιμή 1
  mask = (qa.bitwiseAnd(cloudShadowBitMask).eq(0)  #  eq 0 σημαίνει ότι δεν υπάρχει σκιά σύννεφου
    .And(qa.bitwiseAnd(cloudsBitMask).eq(0))  # eq 0 σημαίνει ότι δεν υπάρχει σύννεφο
    .And(qa.bitwiseAnd(snowsBitMask).eq(0)) # eq 0 σημαίνει ότι δεν υπάρχει χιόνι
    .And(qa.bitwiseAnd(FillBitMask).eq(0)) # eq 0 σημαίνει ότι τα δεδομένα ειναι έγκυρα
    .And(qa.bitwiseAnd(DilatedCloudBitMask).eq(0)) # eq 0 σημαίνει ότι δεν υπάρχει διευρυμένο σύννεφο
    .And(qa.bitwiseAnd(WaterBitMask).eq(0))) # eq 0 σημαίνει ότι δεν υπάρχει νερό
  
  
  saturationMask = image.select('QA_RADSAT').eq(0); # eq 0 σημαίνει ότι δεν υπάρχει κορεσμός ακτινοβολίας (radiometric saturation)
  
           
  # επιστροφή της εικόνας με εφαρμοσμένο το mask για τα σύννεφα, τη σκιά σύννεφου, το χιόνι, το νερό και τα έγκυρα δεδομένα,
  #  καθώς και το mask για τον κορεσμό ακτινοβολίας
  maskedimage = image.updateMask(mask).updateMask(saturationMask)
  
  # προσθήκη της band 'Water' που δείχνει αν το pixel είναι νερό ή όχι, με βάση το bitmask για το νερό στην band QA_PIXEL
  maskedimage = maskedimage.addBands(
    ee.Image([qa.bitwiseAnd(WaterBitMask).eq(0).rename('Water')])
    ).copyProperties(image).set('system:time_start', image.get('system:time_start'));
    
    
  return (maskedimage)

# εφαρμογή της συνάρτησης maskL8sr για την αφαίρεση των νεφών, 
# της σκιάς σύννεφου, του χιονιού, του νερού και των μη έγκυρων δεδομένων, καθώς και των pixels με κορεσμό ακτινοβολίας
L8_scaled_qa = L8_scaled.map(maskL8sr)


Στον παρακάτω χάρτη παρουσιάζεται ενδεικτικά η QA_PIXEL μπάντα για την εικόνα με index = 60 στο ImageCollection:

In [ ]:

# 1. Αρχικοποίηση Χάρτη
Map = geemap.Map()

# 2. Φόρτωση μιας εικόνας Landsat 8 (Collection 2)
# Επιλέγουμε μια περιοχή με σύννεφα για να δούμε ποικιλία τιμών


collection_list = clear_L8.toList(clear_L8.size()) # μετατροπή της συλλογής σε λίστα για να έχουμε πρόσβαση σε συγκεκριμένες εικόνες με βάση τη θέση τους στη λίστα

index = 60
# 2. Επιλογή της εικόνας στη θέση 10
image = ee.Image(collection_list.get(index))


#first_image = clear_L8.first()
qa = image.select('QA_PIXEL')

# 3. Αυτόματη εύρεση Μοναδικών Τιμών (Unique Values)
# Προσοχή: Χρησιμοποιούμε μεγαλύτερο scale για να μην κολλήσει ο υπολογισμός
stats = qa.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=image.geometry(),
    scale=90,
    maxPixels=1e9
)

# Μετατροπή των τιμών σε λίστα ακεραίων
unique_values_dict = stats.get('QA_PIXEL').getInfo()
unique_values = sorted([int(float(k)) for k in unique_values_dict.keys()])
new_values = list(range(len(unique_values)))

# 4. Δημιουργία χρωμάτων (Παλέτα)
# Μπορείς να βάλεις δικά σου χρώματα ή τυχαία
def generate_random_color():
    return '#%06x' % random.randint(0, 0xFFFFFF)

colors = [generate_random_color() for _ in unique_values]

# 5. Δημιουργία της οπτικοποιημένης (Visual) εικόνας
# Με το remap αντιστοιχίζουμε τις μεγάλες τιμές (π.χ. 21824) σε 0, 1, 2...
qa_remapped = qa.remap(unique_values, new_values)

# Με το .visualize() μετατρέπουμε την εικόνα σε RGB (3-band byte image)
# Έτσι τα χρώματα "κλειδώνουν" πάνω στις τιμές
qa_rgb = qa_remapped.visualize(
    min=0, 
    max=len(unique_values) - 1, 
    palette=colors
)

# 6. Προσθήκη Layer στον Χάρτη
Map.centerObject(image, 10)

# Πρώτο Layer: Το οπτικό (για τα χρώματα)
Map.addLayer(qa_rgb, {}, 'QA Visualization (Correct Colors)')

# Δεύτερο Layer: Τα πραγματικά δεδομένα (Κρυφό - για το Inspector)
# Όταν πατάς με το Inspector, θα διαβάζει την τιμή από εδώ
Map.addLayer(qa, {}, 'QA Actual Data (For Inspector)', False)

# 7. Προσθήκη Υπομνήματος (Legend)
legend_dict = {str(val): col for val, col in zip(unique_values, colors)}
Map.add_legend(title="QA_PIXEL Values", legend_dict=legend_dict)

Map

# Οπτικοποίηση χρονοσειράς

στο παρακάτω παράδειγμα προσθέτουμε ένα time slider για να δούμε μια True color image (SR_B4, SR_B3, SR_B2) για τη συλλογή *L8_scaled_qa*


In [ ]:
Map = geemap.Map()
vis_params = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.2}
Map.add_time_slider(L8_scaled_qa, vis_params, time_interval=1)
Map.centerObject(L8_scaled_qa, 8)

Map


# Σύγκριση εικόνων χρονοσειράς

Ακόμα μπορούμε να συγκρίνουμε οπτικά τις εικόνες απο τις χρονοσειρές L8_scaled και L8_scaled_qa:

In [ ]:
Map = geemap.Map()

# Λήψη των timestamps από την ιδιότητα 'system:time_start' για όλες τις εικόνες της συλλογής
timestamps = L8_scaled.aggregate_array('system:time_start').getInfo()

# Μετατροπή των timestamps σε μορφή (YYYY-MM-DD)
date_list = [datetime.datetime.fromtimestamp(ts / 1000).strftime('%Y-%m-%d') for ts in timestamps]


Map.ts_inspector(
    left_ts=L8_scaled,
    right_ts=L8_scaled_qa,
    left_names=date_list,
    right_names=date_list,
    left_vis=vis_params,
    right_vis=vis_params,
    width='100px', # πλάτος των dropdown μενού
)
Map.add_basemap('OpenStreetMap.Mapnik') # list(geemap.basemaps.keys())
Map.centerObject(L8_scaled_qa, 8)
Map


# Clip εικόνων ενός ImageCollection

Για να αποκόψουμε (clip) της εικόνες μιας χρονοσειράς με βάση μια γεωμετρία, πρέπει να κάνουμε ένα βρόγχο μέσω του *map* και για κάθε μια εικόνα να χρησιμοποιήσουμε την μέθοδο clip με όρισμα την γεωμετρία. Αυτό μπορεί να εφαρμοστεί είτε με μια συνάρτηση είτε με μια ανώνυμη συνάρτηση lambda. 

In [ ]:

def clip_to_roi(image):
    return image.clip(attiki_region)

L8_scaled_qa_clipped = L8_scaled_qa.map(clip_to_roi)

# ή εναλλακτικά με lambda function
L8_scaled_qa_clipped = L8_scaled_qa.map(lambda img: img.clip(attiki_region))



# Δημιουργία composite και υπολογισμός δείκτη NDVI

In [ ]:
# Υπολογισμός του διάμεσου (median) για κάθε pixel σε όλες τις εικόνες της συλλογής

median_L8 = L8_scaled_qa.median() # ουστιαστικά είναι συντομογραφία της εκτέλεσης με reducer: median_L8 = L8_scaled_qa.reduce(ee.Reducer.median())

# Υπολογισμός NDVI
# Band 5 = NIR, Band 4 = Red
ndvi = median_L8.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# Clip στην περιοχή ενδιαφέροντος
ndvi = ndvi.clip(attiki_region)

# Εμφάνιση αποτελεσμάτων
Map = geemap.Map()
Map.add_basemap('Esri.WorldShadedRelief') # list(geemap.basemaps.keys())

# Παλέτα για NDVI (από καφέ/κίτρινο σε βαθύ πράσινο)
ndvi_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#FFFFFF', '#CE7E45', '#DF923D', '#F1B555', '#FCD163', 
                '#99B718', '#74A401', '#3E8601', '#207401', '#056201']
}

# True Color Image (Bands 4, 3, 2) με κλίμακα 0-0.2 για καλύτερη εμφάνιση
Map.addLayer(L8_scaled_qa, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.2}, 'True Color Image L8')
Map.addLayer(L8_scaled_qa_clipped, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.2}, 'Clipped', False, 1) # 1=opacity
Map.addLayer(ndvi, ndvi_vis, 'NDVI Single Image')

Map.centerObject(attiki_region, 8)

Map

# Εξαγωγή εικόνων

## Export.image.toDrive
Για την λήψη μιας ολόκληρης συλλογής εικόνων (ImageCollection) από το Google Earth Engine στο Google Drive, μπορεί να χρησιμοποιηθεί η εντολή Export.image.toDrive.

Επειδή η εντολή Export λειτουργεί μόνο για μεμονωμένες εικόνες (ee.Image), η στρατηγική είναι να μετατρέψεις τη συλλογή σε λίστα και να κάνεις "export" κάθε εικόνα ξεχωριστά χρησιμοποιώντας ένα loop. Στο παρακάτω παράδειγμα γίνεται λήψη των εικόνων της συλλογής *L8_scaled_qa*

In [ ]:

# Μετατροπή της συλλογής σε λίστα
col_list = L8_scaled_qa.toList(L8_scaled_qa.size()) # μετατροπή του imagecollection σε μία λίστα από εικόνες
count = L8_scaled_qa.size().getInfo() # το πλήθος των εικόνων

# Loop για την εξαγωγή κάθε εικόνας
# για λόγους εξοικονόμησης χρόνο ας κατεβάσουμε τις πρώτες 5 εικόνες
count = 5
for i in range(count):
    image = ee.Image(col_list.get(i))
    
    # Δημιουργία ονόματος αρχείου (π.χ. από ένα property ή το index)
    # Εδώ χρησιμοποιούμε το property 'month' που ορίσαμε πριν
    month_val = ee.Date(image.get('system:time_start')).get('month').getInfo()
    day = ee.Date(image.get('system:time_start')).get('day').getInfo()
    year_val = image.get('year').getInfo()
    file_name = f'L8_scaled_qa_{year_val}_{month_val}_{day}'
    
    #  Export task
    task = ee.batch.Export.image.toDrive(
        image=image.select(['SR_B4', 'SR_B3', 'SR_B2']), # επιλέγουμε τις μπάντες που μας ενδιαφέρει
        description=file_name,
        folder='GEE_Exports',        # Το όνομα του φακέλου στο Google Drive 
        fileNamePrefix=file_name,
        scale=30,                    # Ανάλυση σε μέτρα (30m για Landsat)
        crs='EPSG:2100',
        region=image.geometry(),     # Η περιοχή κάλυψης
        maxPixels=1e13               # Όριο maxPixels για μεγάλες περιοχές
    )
    
    # Εκκίνηση της εργασίας
    task.start()
    print(f'Started task: {file_name}')

Ο χρήστης έχει την επιλογή να κάνει το ίδιο μεσω το *geemap* και του *ee_export_image_collection_to_drive*

In [ ]:
images_to_export = 5 

geemap.ee_export_image_collection_to_drive(
    L8_scaled_qa.select(['SR_B4', 'SR_B3', 'SR_B2']).limit(images_to_export), 
    folder='GEE_Exports', 
    scale=30, 
    region=attiki_region.geometry(),
    crs='EPSG:2100',
    maxPixels=1e13   
)



Αντί να ανατρέχουμε στο google console μπορούμε να δούμε την κατάσταση από τα tasks μέσω το *ee* και του *listOperations*

In [ ]:
# 
# Επιστρέφει μια λίστα με όλα τα tasks και την κατάστασή τους
tasks = ee.data.listOperations()
for task in tasks:
    print(f"Task: {task['metadata']['description']} - State: {task['metadata']['state']}")

## Λήψη μέσω το ee_export_image_collection του geemap
Το geemap διαθέτει μια σχετική συνάρτηση που αυτοματοποιεί το loop που είδαμε πριν, κατεβάζοντας κάθε εικόνα της συλλογής ως .tif σε έναν φάκελο της επιλογής του χρήστη. Ωστόσο η επιλογή αυτή ενδείκνυται για μικρές σε μέγεθος εικόνες. Σε περιπτώσεις μεγάλων εικόνων σε μέγεθος η συνάρτηση αποτυγχάνει.

In [ ]:
# Χρήση της συνάρτησης του geemap για ee_export_image_collection

geemap.ee_export_image_collection(
    L8_scaled_qa.select(['SR_B4', 'SR_B3', 'SR_B2']).limit(images_to_export), 
    out_dir=OUTPUT_DIR, 
    scale=500, 
    crs='EPSG:2100',
    region=image.geometry(),      # την περιοχή κάλυψης
    file_per_band=False # Αν θες όλες τις μπάντες σε ένα αρχείο ανά εικόνα
)